- Month-wise	count	of	new	rentals	started	and	month-wise	rental	revenue collected	(successful	payments	only),	for	the	full	period	in	the	data.
- Reproduce	your	Part	A,	question	1	result	and	confirm	it	matches	your	SQL output.

In [10]:
import pandas as pd

payments = pd.read_csv("C:/Projects/Cityfurnish-Assignment/data/payments.csv", parse_dates=["payment_date"])
rentals = pd.read_csv("C:/Projects/Cityfurnish-Assignment/data/rentals.csv", parse_dates=["start_date", "end_date"])

In [11]:
# display(customers.info())
# display(payments.info())
# display(rentals.info())

In [12]:
paid = payments[payments["status"]=="success"]

new_rentals = rentals.groupby(rentals["start_date"].dt.to_period("M")).size().rename("new_rentals")
revenue = payments.groupby(paid["payment_date"].dt.to_period("M"))["amount"].sum().rename("revenue")

pandas_output = (pd.concat([new_rentals, revenue], axis=1).fillna(0).rename_axis("month").reset_index())
pandas_output["new_rentals"] = pandas_output["new_rentals"].astype(int)
pandas_output["month"] = pandas_output["month"].dt.to_timestamp()

In [13]:
pandas_output

,month,new_rentals,revenue
0,2026-01-01,51,41457
1,2026-02-01,77,103500
2,2026-03-01,87,165028
3,2026-04-01,50,181018
4,2026-05-01,107,254458
5,2026-06-01,74,275840
6,2026-07-01,61,264048
7,2026-08-01,71,242066
8,2026-09-01,71,216489
9,2026-10-01,11,182129


##### matching pandas output with sql output

In [14]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass
import psycopg2

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=getpass("DB password: "),
    host="localhost",
    port=5432,
    database="cityfurnish",
))

with open("C:/Projects/Cityfurnish-Assignment/part_a_sql/question_1.sql") as f:
    sql_result = pd.read_sql(text(f.read()), engine)

In [15]:
sql_result.to_csv("sql_output.csv", index=False)

sql_result["month"] = pd.to_datetime(sql_result["month"])
sql_result["revenue"] = sql_result["revenue"].astype(int)

display(sql_result)

,month,new_rentals,revenue
0,2026-01-01,51,41457
1,2026-02-01,77,103500
2,2026-03-01,87,165028
3,2026-04-01,50,181018
4,2026-05-01,107,254458
5,2026-06-01,74,275840
6,2026-07-01,61,264048
7,2026-08-01,71,242066
8,2026-09-01,71,216489
9,2026-10-01,11,182129


In [16]:
pd.testing.assert_frame_equal(pandas_output, sql_result, check_dtype=False, atol=0.01)
print("MATCH: pandas result equals SQL output")

MATCH: pandas result equals SQL output
